# Etapa 2 — Experimentos e Escolha do Melhor Modelo

**Problema:** Regressão  
**Métricas:** MAE e R²  
**Experimentos:** 30 execuções por modelo (média e desvio padrão)

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
df = pd.read_csv('CAR DETAILS FROM CAR DEKHO.csv')
df = df.drop_duplicates().dropna()
df = df[(df['selling_price'] > 0) & (df['year'] >= 1990)]

df['brand'] = df['name'].apply(lambda x: str(x).split()[0])
df = df.drop('name', axis=1)

df_encoded = pd.get_dummies(df, columns=['brand','fuel','seller_type','transmission','owner'])
X = df_encoded.drop('selling_price', axis=1)
y = df_encoded['selling_price']

print('Features:', X.shape[1])
print('Amostras:', X.shape[0])

## Experimentos (30 runs por modelo)

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42, max_depth=15),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'SVR': Pipeline([('scaler', StandardScaler()), ('svr', SVR(kernel='rbf', C=100))])
}

results = {name: {'mae': [], 'r2': []} for name in models}

for seed in range(30):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        results[name]['mae'].append(mean_absolute_error(y_test, pred))
        results[name]['r2'].append(r2_score(y_test, pred))

summary = []
for name, metrics in results.items():
    summary.append({
        'Modelo': name,
        'MAE Médio': np.mean(metrics['mae']),
        'MAE Std': np.std(metrics['mae']),
        'R2 Médio': np.mean(metrics['r2']),
        'R2 Std': np.std(metrics['r2'])
    })

pd.DataFrame(summary).sort_values('R2 Médio', ascending=False)

## Justificativa da Escolha

O **Random Forest** apresentou o maior R² médio e menor MAE médio nas 30 execuções, além de menor variabilidade entre runs em comparação com a Árvore de Decisão isolada.

Modelos lineares capturam relações simples, enquanto o Random Forest lida melhor com interações não lineares entre marca, ano e quilometragem.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

best_model = RandomForestRegressor(n_estimators=100, random_state=42)
best_model.fit(X_train, y_train)

pred = best_model.predict(X_test)
print('MAE final:', mean_absolute_error(y_test, pred))
print('R2 final:', r2_score(y_test, pred))

joblib.dump(best_model, 'car_price_model.pkl')
joblib.dump(list(X.columns), 'model_columns.pkl')
print('Modelo exportado com sucesso!')

## Papéis da Equipe (editar com os nomes reais)

| Membro | Responsabilidade |
|--------|------------------|
| Membro 1 | Coleta de dados e análise descritiva |
| Membro 2 | Pré-processamento e experimentos ML |
| Membro 3 | Backend (FastAPI) e banco de dados |
| Membro 4 | Frontend e documentação |